<a href="https://colab.research.google.com/github/HARSHUL-1312/fake-news-prediction/blob/main/fake_news_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import re
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [ ]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
print(stopwords.words('english'))

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

In [ ]:
news_dataset=pd.read_csv('/content/fake_news_dataset.csv')

In [ ]:
news_dataset.shape

(20000, 7)

In [ ]:
news_dataset.head()

,title,text,date,source,author,category,label
0,Foreign Democrat final.,more tax development both store agreement lawy...,2023-03-10,NY Times,Paula George,Politics,real
1,To offer down resource great point.,probably guess western behind likely next inve...,2022-05-25,Fox News,Joseph Hill,Politics,fake
2,Himself church myself carry.,them identify forward present success risk sev...,2022-09-01,CNN,Julia Robinson,Business,fake
3,You unit its should.,phone which item yard Republican safe where po...,2023-02-07,Reuters,Mr. David Foster DDS,Science,fake
4,Billion believe employee summer how.,wonder myself fact difficult course forget exa...,2023-04-03,CNN,Austin Walker,Technology,fake


In [ ]:
news_dataset.describe()

,title,text,date,source,author,category,label
count,20000,20000,20000,19000,19000,20000,20000
unique,20000,20000,1096,8,17051,7,2
top,Turn present write decision town human personal.,suffer tree increase prevent organization easy...,2023-08-31,Daily News,Michael Smith,Health,fake
freq,1,1,32,2439,12,2922,10056


In [ ]:
news_dataset.isnull().sum()

,0
title,0
text,0
date,0
source,1000
author,1000
category,0
label,0


In [ ]:
news_dataset=news_dataset.fillna('')

In [ ]:
news_dataset['contet']=news_dataset['title']+''+news_dataset['author']+''+news_dataset['source']+''+news_dataset['category']

In [ ]:
print(news_dataset['contet'])

0        Foreign Democrat final.Paula GeorgeNY TimesPol...
1        To offer down resource great point.Joseph Hill...
2        Himself church myself carry.Julia RobinsonCNNB...
3        You unit its should.Mr. David Foster DDSReuter...
4        Billion believe employee summer how.Austin Wal...
                               ...                        
19995          House party born.Gary MilesBBCEntertainment
19996    Though nation people maybe price box.Maria Mcb...
19997    Yet exist with experience unit.Kristen Frankli...
19998      School wide itself item.David WiseReutersHealth
19999    Offer chair cover senior born.James PetersonDa...
Name: contet, Length: 20000, dtype: object


In [ ]:
x=news_dataset.drop('label',axis=1)
y=news_dataset['label']

In [ ]:
print(x)
print(y)

                                       title  ...                                             contet
0                    Foreign Democrat final.  ...  Foreign Democrat final.Paula GeorgeNY TimesPol...
1        To offer down resource great point.  ...  To offer down resource great point.Joseph Hill...
2               Himself church myself carry.  ...  Himself church myself carry.Julia RobinsonCNNB...
3                       You unit its should.  ...  You unit its should.Mr. David Foster DDSReuter...
4       Billion believe employee summer how.  ...  Billion believe employee summer how.Austin Wal...
...                                      ...  ...                                                ...
19995                      House party born.  ...        House party born.Gary MilesBBCEntertainment
19996  Though nation people maybe price box.  ...  Though nation people maybe price box.Maria Mcb...
19997        Yet exist with experience unit.  ...  Yet exist with experience unit.Kristen F

In [ ]:
port_stem=PorterStemmer()

In [ ]:
def stemming(content):
  stemmed_content=re.sub('[^a-zA-Z]',' ',content)
  stemmed_content=stemmed_content.lower()
  stemmed_content=stemmed_content.split()
  stemmed_content=[port_stem.stem(word) for word in stemmed_content if not word in stopwords.words('english')]
  stemmed_content=' '.join(stemmed_content)
  return stemmed_content

In [ ]:
news_dataset['contet']=news_dataset['contet'].apply(stemming)

In [ ]:
print(news_dataset['contet'])

0         foreign democrat final paula georgeni timespolit
1        offer resourc great point joseph hillfox newsp...
2                       church carri julia robinsoncnnbusi
3                       unit mr david foster ddsreuterssci
4        billion believ employe summer austin walkercnn...
                               ...                        
19995               hous parti born gari milesbbcentertain
19996    though nation peopl mayb price box maria mcbri...
19997    yet exist experi unit kristen franklinbbcenter...
19998             school wide item david wisereutershealth
19999    offer chair cover senior born jame petersondai...
Name: contet, Length: 20000, dtype: object


In [ ]:
x=news_dataset['contet'].values
y=news_dataset['label'].values

In [ ]:
print(y)

['real' 'fake' 'fake' ... 'real' 'fake' 'fake']


In [ ]:
y.shape

(20000,)

In [ ]:
vectorizer=TfidfVectorizer()
vectorizer.fit(x)
x=vectorizer.transform(x)

In [ ]:
print(x)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 149697 stored elements and shape (20000, 11036)>
  Coords	Values
  (0, 2503)	0.3434743805417029
  (0, 3050)	0.3363622660941335
  (0, 3200)	0.3753525271475403
  (0, 3534)	0.576478490621184
  (0, 7617)	0.46962167537823396
  (0, 9919)	0.2740881027872787
  (1, 3797)	0.3645834167438975
  (1, 4459)	0.5295474160891566
  (1, 5166)	0.3552640546715987
  (1, 7151)	0.2619741539861179
  (1, 7329)	0.36316015192729334
  (1, 7885)	0.3636308329300718
  (1, 8275)	0.3573552599933625
  (2, 1544)	0.4034244282665368
  (2, 1807)	0.41304894426665684
  (2, 5191)	0.5068640909591784
  (2, 8558)	0.6401000653133413
  (3, 2338)	0.3158543190605834
  (3, 2441)	0.5806920022551827
  (3, 3205)	0.5806920022551827
  (3, 6982)	0.29293042401352176
  (3, 10077)	0.37419435171013377
  (4, 375)	0.42282660882577083
  (4, 724)	0.3477656383381861
  (4, 848)	0.3524086077065066
  :	:
  (19996, 6225)	0.5007534749627158
  (19996, 7089)	0.28002585513000294
  (19996, 7148)	0.

In [ ]:
x_train,X_test,y_train,y_test=train_test_split(x,y,test_size=0.2,stratify=y,random_state=2)

In [ ]:
model=LogisticRegression()

In [ ]:
model.fit(x_train,y_train)

LogisticRegression()

In [ ]:
x_train_prediction=model.predict(x_train)
training_data_accuracy=accuracy_score(x_train_prediction,y_train)

In [ ]:
print('accuracy score of tranning data',training_data_accuracy)

accuracy score of tranning data 0.7206875


In [ ]:
x_test_prediction=model.predict(X_test)
test_data_accuracy=accuracy_score(x_test_prediction,y_test)

In [ ]:
print('accuracy score of test data ',test_data_accuracy)

accuracy score of test data  0.49175


In [ ]:
x_new=X_test[6]
prediction=model.predict(x_new)
print(prediction)
if (prediction[0]=='real'):
  print('the news is real')
else:
  print('the news is fake')

['real']
the news is real


In [ ]:
print(y_test[6])

real
